In [1]:
#import python libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
#import data
df = pd.read_csv('CarPrice_Assignment.csv')

In [9]:
import re
from difflib import SequenceMatcher
import pandas as pd

def normalize_name(value):
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

car_names = df["CarName"].dropna().astype(str)
unique_names = car_names.unique()

# Print every distinct CarName with its position in the unique list.
for index, name in enumerate(unique_names, start=1):
    print(f"{index:3}. {name}")

# Find likely brand/model spelling variants.
comparison_rows = []

for i, name_i in enumerate(unique_names):
    normalized_i = normalize_name(name_i)

    for j in range(i + 1, len(unique_names)):
        name_j = unique_names[j]
        normalized_j = normalize_name(name_j)

        if normalized_i == normalized_j:
            continue

        similarity = SequenceMatcher(None, normalized_i, normalized_j).ratio()

        # A high score indicates very similar spelling.
        if similarity >= 0.82:
            comparison_rows.append({
                "car_name_1": name_i,
                "car_name_2": name_j,
                "similarity": round(similarity * 100, 2)
            })

pd.DataFrame(comparison_rows).sort_values(
    "similarity",
    ascending=False
).to_string(index=False)

  1. alfa romeo giulia
  2. alfa romeo stelvio
  3. alfa romeo Quadrifoglio
  4. audi 100 ls
  5. audi 100ls
  6. audi fox
  7. audi 5000
  8. audi 4000
  9. audi 5000s (diesel)
 10. bmw 320i
 11. bmw x1
 12. bmw x3
 13. bmw z4
 14. bmw x4
 15. bmw x5
 16. chevrolet impala
 17. chevrolet monte carlo
 18. chevrolet vega 2300
 19. dodge rampage
 20. dodge challenger se
 21. dodge d200
 22. dodge monaco (sw)
 23. dodge colt hardtop
 24. dodge colt (sw)
 25. dodge coronet custom
 26. dodge dart custom
 27. dodge coronet custom (sw)
 28. honda civic
 29. honda civic cvcc
 30. honda accord cvcc
 31. honda accord lx
 32. honda civic 1500 gl
 33. honda accord
 34. honda civic 1300
 35. honda prelude
 36. honda civic (auto)
 37. isuzu MU-X
 38. isuzu D-Max 
 39. isuzu D-Max V-Cross
 40. jaguar xj
 41. jaguar xf
 42. jaguar xk
 43. mazda rx3
 44. mazda glc deluxe
 45. mazda rx2 coupe
 46. mazda rx-4
 47. mazda 626
 48. mazda glc
 49. mazda rx-7 gs
 50. mazda glc 4
 51. mazda glc custom l
 52. ma

'             car_name_1                car_name_2  similarity\n            audi 100 ls                audi 100ls       95.24\n              saab 99le                saab 99gle       94.74\n               saab 99e                 saab 99le       94.12\n     mazda glc custom l          mazda glc custom       94.12\ntoyota corolla liftback    toyota corona liftback       93.33\n   dodge coronet custom dodge coronet custom (sw)       93.02\n      mitsubishi mirage      mitsubishi mirage g4       91.89\n            peugeot 504               peugeot 304       90.91\n              mazda glc               mazda glc 4       90.00\n              jaguar xf                 jaguar xk       88.89\n          toyota corona            toyota corolla       88.89\n              subaru r1                 subaru r2       88.89\n               saab 99e                saab 99gle       88.89\n              audi 5000                 audi 4000       88.89\n              volvo 245                 volvo 246     

In [10]:
matches = pd.DataFrame(comparison_rows)

if matches.empty:
    print("No name pairs met the 82% similarity threshold.")
else:
    display(matches.sort_values("similarity", ascending=False))

,car_name_1,car_name_2,similarity
0,audi 100 ls,audi 100ls,95.24
30,saab 99le,saab 99gle,94.74
28,saab 99e,saab 99le,94.12
18,mazda glc custom l,mazda glc custom,94.12
38,toyota corolla liftback,toyota corona liftback,93.33
9,dodge coronet custom,dodge coronet custom (sw),93.02
20,mitsubishi mirage,mitsubishi mirage g4,91.89
24,peugeot 504,peugeot 304,90.91
17,mazda glc,mazda glc 4,90.00
15,jaguar xf,jaguar xk,88.89


In [4]:
df["CarName"].nunique()

147

In [5]:
df["CarName"].nunique

<bound method IndexOpsMixin.nunique of 0            alfa-romero giulia
1           alfa-romero stelvio
2      alfa-romero Quadrifoglio
3                   audi 100 ls
4                    audi 100ls
                 ...           
200             volvo 145e (sw)
201                 volvo 144ea
202                 volvo 244dl
203                   volvo 246
204                 volvo 264gl
Name: CarName, Length: 205, dtype: object>

In [6]:
# Map common brand misspellings to the correct brand names.
brand_corrections = {
    "alfa-romero": "alfa romeo",
    "chevrollet": "chevrolet",
    "maxda": "mazda",
    "porcshce": "porsche",
    "toyouta": "toyota",
    "vokswagen": "volkswagen",
    "vw": "volkswagen",
}

# Normalise the dictionary keys so they match the output
# of normalize_name().
brand_corrections = {
    normalize_name(key): value
    for key, value in brand_corrections.items()
}

def correct_brand(value):
    if pd.isna(value):
        return value

    parts = str(value).split(maxsplit=1)

    # The first whitespace-separated token is the brand.
    brand = normalize_name(parts[0])
    corrected_brand = brand_corrections.get(brand, parts[0])

    # Preserve the model name for now.
    if len(parts) > 1:
        return f"{corrected_brand} {parts[1]}"

    return corrected_brand

df["CarName"] = df["CarName"].apply(correct_brand)

In [7]:
print(df["CarName"].unique())

['alfa romeo giulia' 'alfa romeo stelvio' 'alfa romeo Quadrifoglio'
 'audi 100 ls' 'audi 100ls' 'audi fox' 'audi 5000' 'audi 4000'
 'audi 5000s (diesel)' 'bmw 320i' 'bmw x1' 'bmw x3' 'bmw z4' 'bmw x4'
 'bmw x5' 'chevrolet impala' 'chevrolet monte carlo' 'chevrolet vega 2300'
 'dodge rampage' 'dodge challenger se' 'dodge d200' 'dodge monaco (sw)'
 'dodge colt hardtop' 'dodge colt (sw)' 'dodge coronet custom'
 'dodge dart custom' 'dodge coronet custom (sw)' 'honda civic'
 'honda civic cvcc' 'honda accord cvcc' 'honda accord lx'
 'honda civic 1500 gl' 'honda accord' 'honda civic 1300' 'honda prelude'
 'honda civic (auto)' 'isuzu MU-X' 'isuzu D-Max ' 'isuzu D-Max V-Cross'
 'jaguar xj' 'jaguar xf' 'jaguar xk' 'mazda rx3' 'mazda glc deluxe'
 'mazda rx2 coupe' 'mazda rx-4' 'mazda 626' 'mazda glc' 'mazda rx-7 gs'
 'mazda glc 4' 'mazda glc custom l' 'mazda glc custom'
 'buick electra 225 custom' 'buick century luxus (sw)' 'buick century'
 'buick skyhawk' 'buick opel isuzu deluxe' 'buick skylark

In [8]:
df["CarName"].nunique

<bound method IndexOpsMixin.nunique of 0            alfa romeo giulia
1           alfa romeo stelvio
2      alfa romeo Quadrifoglio
3                  audi 100 ls
4                   audi 100ls
                ...           
200            volvo 145e (sw)
201                volvo 144ea
202                volvo 244dl
203                  volvo 246
204                volvo 264gl
Name: CarName, Length: 205, dtype: object>